# Google - The Gemma 4 Developer Agent Competition
## Notebook 01: Dataset Inspection & Model Card Launcher

This notebook acts as a launcher to:
1. Import the modular `pipeline` package (from local directory or Kaggle dataset).
2. Load and inspect the competition dataset (`tasks.jsonl`).
3. Load the **Gemma 4 Model Card** and inspect tokenizers, chat templates, and context lengths.
4. Prepare and format trajectory samples for LoRA fine-tuning.

In [ ]:
# ---------------------------------------------------------------------------
# Pipeline Import Bootstrap (Compatible with both Local and Kaggle Datasets)
# ---------------------------------------------------------------------------
import os
import sys
from pathlib import Path

# Detect environment and resolve pipeline location
LOCAL_ROOT = Path("..").resolve()
KAGGLE_INPUT = Path("/kaggle/input")

candidate_paths = [
    LOCAL_ROOT,
    LOCAL_ROOT / "pipeline",
    Path("."),
    Path("pipeline")
]

# Check Kaggle dataset mounts
if KAGGLE_INPUT.exists():
    for dataset_dir in KAGGLE_INPUT.glob("*"):
        if (dataset_dir / "pipeline").exists():
            candidate_paths.insert(0, dataset_dir)
        elif (dataset_dir / "main.py").exists() and "pipeline" in dataset_dir.name.lower():
            candidate_paths.insert(0, dataset_dir.parent)

pipeline_loaded = False
for p in candidate_paths:
    if (p / "pipeline" / "main.py").exists() or (p / "pipeline").exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        print(f"Successfully loaded pipeline from: {p}")
        pipeline_loaded = True
        break

if not pipeline_loaded:
    print("[WARN] Pipeline folder not auto-detected; defaulting to current sys.path.")

from pipeline.main import PipelineRunner, status, train, evaluate, package
runner = PipelineRunner()
runner.status()

### 1. Load & Inspect Dataset using the Pipeline Loader
We import `load_tasks` directly from `pipeline.src.data`.

In [ ]:
from pipeline.src.data.loader import load_tasks

# Load benchmark tasks (from local data/ or Kaggle competition input)
tasks_path = Path("../data/val_tasks.jsonl")
if not tasks_path.exists():
    kaggle_task_candidate = Path("/kaggle/input/competitions/gemma-4-developer-agent/tasks.jsonl")
    if kaggle_task_candidate.exists():
        tasks_path = kaggle_task_candidate

tasks = load_tasks(tasks_path)
print(f"Loaded {len(tasks)} tasks.")

if tasks:
    sample_task = tasks[0]
    print("\nSample Task Overview:")
    print(f"  • Task ID: {sample_task.get('task_id')}")
    print(f"  • Repo:    {sample_task.get('repo')}")
    print(f"  • Commit:  {sample_task.get('base_commit')}")
    print(f"  • Issue:   {sample_task.get('issue_description', '')[:200]}...")

### 2. Gemma 4 Model Card Exploration
Reviewing architecture, context length (128K/256K), and quantization attributes for:
- `google/gemma-4-12b-it`
- `google/gemma-4-31b-it-qat-w4a16-ct`
- `google/gemma-4-26b-moe`

In [ ]:
print("""
Gemma 4 Developer Agent Model Card:
- Base Variant: google/gemma-4-12b-it
- Context Length: 128,000 tokens (expands up to 256,000)
- Control Tokens:
    Turn Start: <start_of_turn>
    Turn End:   <end_of_turn>
    Roles:      user, model
- Tool Calling: Native function call generation via structured JSON
- Quantization Profile: QAT W4A16 for high-speed single-GPU serving
""")

### 3. Trajectory Processing
Format sample multi-turn dialogues into Gemma 4 token strings using `format_trajectory_for_sft`.

In [ ]:
from pipeline.src.data.processor import format_trajectory_for_sft

sample_messages = [
    {"role": "user", "content": "Fix KeyError in config loader."},
    {"role": "model", "content": "Locating config.py...", "tool_calls": [{"function": {"name": "read_file", "arguments": "{\"path\": \"config.py\"}"}}]},
    {"role": "tool", "name": "read_file", "content": "return data['debug']"},
    {"role": "model", "content": "Editing config.py...", "tool_calls": [{"function": {"name": "edit_file", "arguments": "{\"path\": \"config.py\"}"}}]}
]

formatted_text = format_trajectory_for_sft(sample_messages)
print("Formatted Training Record:\n")
print(formatted_text)